In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
  Write a program to invert the colors of an image. The image is
  represented as a 1D array of RGBA (Red, Green, Blue, Alpha) values, where each
  component is an 8-bit unsigned integer (<code>unsigned char</code>).
</p>

<p>
  Color inversion is performed by subtracting each color component (R, G, B)
  from 255. The Alpha component should remain unchanged.
</p>

<p>
  The input array
  <code>image</code> will contain <code>width * height * 4</code> elements. The
  first 4 elements represent the RGBA values of the top-left pixel, the next 4
  elements represent the pixel to its right, and so on.
</p>

<h2>Implementation Requirements</h2>
<ul>
  <li>Use only native features (external libraries are not permitted)</li>
  <li>The <code>solve</code> function signature must remain unchanged</li>
  <li>
    The final result must be stored in the array
    <code>image</code>
  </li>
</ul>

<h2>Example 1:</h2>
<pre>
Input: image = [255, 0, 128, 255, 0, 255, 0, 255], width=1, height=2
Output: [0, 255, 127, 255, 255, 0, 255, 255]
</pre>

<h2>Example 2:</h2>
<pre>
Input: image = [10, 20, 30, 255, 100, 150, 200, 255], width=2, height=1
Output: [245, 235, 225, 255, 155, 105, 55, 255]
</pre>

<h2>Constraints</h2>

<ul>
  <li>1 &le; <code>width</code> &le; 4096</li>
  <li>1 &le; <code>height</code> &le; 4096</li>
  <li><code>width * height</code> &le; 8,388,608.</li>

  <li>Performance is measured with <code>height</code> = 5,120, <code>width</code> = 4,096</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

__global__ void invert_kernel(unsigned char* image, int width, int height) {}
// image_input, image_output are device pointers (i.e. pointers to memory on the GPU)
extern "C" void solve(unsigned char* image, int width, int height) {
    int threadsPerBlock = 256;
    int blocksPerGrid = (width * height + threadsPerBlock - 1) / threadsPerBlock;

    invert_kernel<<<blocksPerGrid, threadsPerBlock>>>(image, width, height);
    cudaDeviceSynchronize();
}


# CUTE

In [ ]:
%%writefile solution.py
import cutlass
import cutlass.cute as cute


# image are tensors on the GPU
@cute.jit
def solve(image: cute.Tensor, width: cute.Int32, height: cute.Int32):
    pass


# JAX

In [ ]:
%%writefile solution.py
import jax
import jax.numpy as jnp


# image is a tensor on the GPU
@jax.jit
def solve(image: jax.Array, width: int, height: int) -> jax.Array:
    # return output tensor directly
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.gpu import block_dim, block_idx, thread_idx
from std.memory import UnsafePointer
from std.math import ceildiv


def invert_kernel(image: UnsafePointer[UInt8, MutExternalOrigin], width: Int32, height: Int32):
    pass


# image is a device pointer (i.e. pointer to memory on the GPU)
@export
def solve(image: UnsafePointer[UInt8, MutExternalOrigin], width: Int32, height: Int32) raises:
    var threadsPerBlock: Int32 = 256
    var ctx = DeviceContext()

    var total_pixels = width * height
    var blocksPerGrid = ceildiv(total_pixels, threadsPerBlock)

    var _kernel = ctx.compile_function[invert_kernel, invert_kernel]()
    ctx.enqueue_function(
        _kernel, image, width, height, grid_dim=blocksPerGrid, block_dim=threadsPerBlock
    )

    ctx.synchronize()


# Torch

In [ ]:
%%writefile solution.py
import torch


# image is a tensor on the GPU
def solve(image: torch.Tensor, width: int, height: int):
    pass


# Triton

In [ ]:
%%writefile solution.py
import torch
import triton
import triton.language as tl


@triton.jit
def invert_kernel(image, width, height, BLOCK_SIZE: tl.constexpr):
    pass


# image is a tensor on the GPU
def solve(image: torch.Tensor, width: int, height: int):
    BLOCK_SIZE = 1024
    n_pixels = width * height
    grid = (triton.cdiv(n_pixels, BLOCK_SIZE),)

    invert_kernel[grid](image, width, height, BLOCK_SIZE)


# Evaluate Setup

In [ ]:
# Download required files from GitHub
!mkdir -p core
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/core/challenge_base.py -O core/challenge_base.py
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/core/evaluator.py -O core/evaluator.py
!wget -q https://raw.githubusercontent.com/lekhit/leetgpu-challenges/main/challenges/easy/7_color_inversion/challenge.py -O challenge.py

from challenge import Challenge
from core.evaluator import Evaluate

ch = Challenge()


# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
